# 02 - CustomOp 注册系统：将 Triton Kernel 集成到 vLLM

> **本 Notebook 涵盖内容**
> - vLLM 的 CustomOp 基类详解
> - 通过 `@CustomOp.register` 注册自定义算子
> - 实现 `forward_native` 与 `forward_cuda` 双路径
> - 平台自动分发机制
> - 编译配置与算子开关

**运行示例**：将上一章的 SiLU Triton kernel 注册为 vLLM CustomOp。

## 1. 没有 CustomOp 时的痛点

假设你写了一个 Triton kernel，想在 vLLM 的模型中使用它：

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import triton
import triton.language as tl

device = torch.device("cuda")

# 你写了一个很棒的 Triton SiLU kernel
@triton.jit
def my_silu_kernel(x_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask).to(tl.float32)
    result = x * tl.sigmoid(x)
    tl.store(output_ptr + offsets, result.to(x_ptr.dtype.element_ty), mask=mask)

def my_silu(x):
    output = torch.empty_like(x)
    n = x.numel()
    my_silu_kernel[(triton.cdiv(n, 1024),)](x, output, n, BLOCK_SIZE=1024)
    return output

# 问题 1: 直接在模型中调用？
class NaiveModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4096, 4096)

    def forward(self, x):
        x = self.linear(x)
        return my_silu(x)  # 直接调用 — 没有平台兼容性!

model = NaiveModel().to(device)
x = torch.randn(128, 4096, device=device, dtype=torch.float16)

# 它能工作......但有三个大问题:
print("1. 没有 CPU fallback — 在 CPU 上会崩溃")
print("2. 没有 torch.compile 兼容 — 编译器无法优化")
print("3. 没有 AMD ROCm 支持 — 不可移植")
print("4. 没有统一的开关 — 无法全局禁用")

## 2. CustomOp 基类：优雅的解决方案

vLLM 的 `CustomOp` 基类解决了上述所有问题。让我们从零重建它的核心逻辑：

In [ ]:
# 为了教学目的，我们重建 CustomOp 的核心逻辑
# 真实版本在 vllm/model_executor/custom_op.py

# ---- 全局注册表 ----
op_registry = {}      # 内部算子注册表
op_registry_oot = {}  # 外部算子注册表

class SimpleCustomOp(nn.Module):
    """
    简化版 CustomOp，展示核心设计思想。

    核心设计:
    1. 每个算子提供 forward_native 和 forward_cuda 两种实现
    2. 构造时自动选择当前平台的实现绑定到 _forward_method
    3. forward() 只是转发到 _forward_method

    > Source: vllm/model_executor/custom_op.py:103-207
    """

    def __new__(cls, *args, **kwargs):
        """创建实例时检查是否有 OOT 替换"""
        op_name = cls.__name__

        # 如果有外部注册的替换实现，使用替换版本
        if op_name in op_registry_oot:
            replacement_cls = op_registry_oot[op_name]
            print(f"  Using OOT replacement: {replacement_cls.__name__}")
            return super().__new__(replacement_cls)

        return super().__new__(cls)

    def __init__(self):
        super().__init__()
        # 在构造时就确定用哪个实现，避免每次 forward 都做分发
        self._forward_method = self._dispatch_forward()

    def forward(self, *args, **kwargs):
        return self._forward_method(*args, **kwargs)

    def _dispatch_forward(self):
        """根据当前平台选择实现"""
        if torch.cuda.is_available():
            return self.forward_cuda
        else:
            return self.forward_native

    def forward_native(self, *args, **kwargs):
        """PyTorch 原生实现 — 必须覆盖"""
        raise NotImplementedError

    def forward_cuda(self, *args, **kwargs):
        """CUDA/Triton 实现 — 必须覆盖"""
        raise NotImplementedError

    # ---- 注册装饰器 ----
    @classmethod
    def register(cls, name):
        """注册内部算子"""
        def decorator(op_cls):
            assert name not in op_registry, f"Duplicate: {name}"
            op_cls.name = name
            op_registry[name] = op_cls
            return op_cls
        return decorator

    @classmethod
    def register_oot(cls, name=None):
        """注册外部替换算子"""
        def decorator(op_cls):
            reg_name = name or op_cls.__name__
            op_registry_oot[reg_name] = op_cls
            return op_cls
        return decorator

print("SimpleCustomOp defined with register/register_oot decorators")

### 2.1 关键设计决策

![CustomOp 分发机制](../diagrams/02-dispatch-mechanism.svg)

**为什么在构造时而非调用时分发？** 因为 LLM 推理会调用 forward() 数万次（每个 token 生成都会调用）。如果每次都做 if/else 检查平台，这些分支判断累积起来是不可忽略的开销。vLLM 的做法是在 `__init__` 时就绑定好方法指针，之后每次调用零开销。

> **Source**: vllm/model_executor/custom_op.py:130-136

## 3. 实战：注册我们的 SiLU 算子

In [ ]:
@SimpleCustomOp.register("my_silu")
class MySiLU(SimpleCustomOp):
    """
    我们的自定义 SiLU 激活函数，同时提供:
    - forward_native: PyTorch 实现（CPU 可用，torch.compile 可优化）
    - forward_cuda: Triton 实现（GPU 高性能）
    """

    def forward_native(self, x: torch.Tensor) -> torch.Tensor:
        """PyTorch-native 实现"""
        return F.silu(x)

    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        """Triton 实现"""
        output = torch.empty_like(x)
        n = x.numel()
        my_silu_kernel[(triton.cdiv(n, 1024),)](
            x, output, n, BLOCK_SIZE=1024
        )
        return output


# 测试
op = MySiLU()
print(f"Op name: {op.name}")
print(f"Registered in op_registry: {'my_silu' in op_registry}")
print(f"Dispatch target: {op._forward_method.__name__}")

x = torch.randn(128, 4096, device=device, dtype=torch.float16)

# 测试 Triton 路径
result = op(x)
expected = F.silu(x)
max_err = (result - expected).abs().max().item()
print(f"\nMax error (Triton vs PyTorch): {max_err:.2e}")
assert max_err < 1e-2, f"Error too large: {max_err}"
print("MySiLU CustomOp: PASSED!")

## 4. 真实案例：vLLM 的 SiluAndMul

现在让我们看看 vLLM 实际的 `SiluAndMul` 是怎么注册的：

```python
# vllm/model_executor/layers/activation.py:117-148

@CustomOp.register("silu_and_mul")
class SiluAndMul(CustomOp):
    """SwiGLU 的激活函数。
    计算 x -> silu(x[:d]) * x[d:]，其中 d = x.shape[-1] // 2

    Shapes:
        x: (num_tokens, 2 * d)
        return: (num_tokens, d)
    """

    def __init__(self, *, compile_native: bool = True):
        super().__init__(compile_native=compile_native)
        if current_platform.is_cuda_alike() or current_platform.is_xpu():
            self.op = torch.ops._C.silu_and_mul  # CUDA C++ kernel
        elif current_platform.is_cpu():
            self._forward_method = self.forward_native

    @staticmethod
    def forward_native(x: torch.Tensor) -> torch.Tensor:
        d = x.shape[-1] // 2
        return F.silu(x[..., :d]) * x[..., d:]

    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        d = x.shape[-1] // 2
        output_shape = x.shape[:-1] + (d,)
        out = torch.empty(output_shape, dtype=x.dtype, device=x.device)
        self.op(out, x)  # 调用 CUDA C++ kernel
        return out
```

注意这里的 `SiluAndMul` 的 `forward_cuda` 实际上调用的是 C++ CUDA kernel（`torch.ops._C.silu_and_mul`），而不是 Triton。但另一个变体 `SwigluStepAndMul` 则使用了 Triton：

```python
@CustomOp.register("swiglustep_and_mul")
class SwigluStepAndMul(CustomOp):
    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        ...
        swiglustep_and_mul_triton(out, x, self.limit)  # Triton kernel!
        return out
```

> **Source**: vllm/model_executor/layers/activation.py:117-148, 341-372

## 5. 实现 SiluAndMul 的 Triton 版本

SwiGLU 是现代 LLM（如 Llama）中最常用的激活函数。它将输入分成两半，一半做 SiLU 激活，另一半作为门控：

$$\text{SwiGLU}(x) = \text{SiLU}(x_{[:d]}) \odot x_{[d:]}$$

其中 $d = \text{x.shape[-1]} / 2$

**数值示例**：设 x = [1.0, 2.0, 3.0, 4.0]（d=2）
- gate = [1.0, 2.0], up = [3.0, 4.0]
- SiLU(gate) = [1.0 * sigmoid(1.0), 2.0 * sigmoid(2.0)] = [0.731, 1.762]
- SwiGLU = [0.731 * 3.0, 1.762 * 4.0] = [2.193, 7.048]

**生活类比**：SwiGLU 就像一个双通道管道系统——一个通道（gate）决定「开多大的阀门」，另一个通道（up）决定「流过多少内容」。最终输出是两者的乘积。

In [ ]:
@triton.jit
def silu_and_mul_kernel(
    output_ptr,
    o_stride,     # 输出的行步长
    input_ptr,
    i_stride,     # 输入的行步长
    d,            # 输出维度 = 输入维度 / 2
    BLOCK_SIZE: tl.constexpr,
):
    """
    SwiGLU 激活函数的 Triton 实现

    对应 vLLM 中的 _swiglustep_and_mul_kernel（activation.py:27-52）
    但简化了 clamping 逻辑。
    """
    # 2D grid: (batch, col_blocks)
    row = tl.program_id(axis=0)  # batch 维度
    col_block = tl.program_id(axis=1)  # hidden_size 块

    offsets = col_block * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < d

    # 计算输入指针
    input_row = input_ptr + row * i_stride
    output_row = output_ptr + row * o_stride

    # 加载 gate 和 up（输入的前半和后半）
    gate = tl.load(input_row + offsets, mask=mask).to(tl.float32)
    up = tl.load(input_row + offsets + d, mask=mask).to(tl.float32)

    # SiLU(gate) * up
    gate_silu = tl.sigmoid(gate) * gate
    result = gate_silu * up

    # 存储结果
    tl.store(output_row + offsets, result.to(input_ptr.dtype.element_ty), mask=mask)


def triton_silu_and_mul(x: torch.Tensor) -> torch.Tensor:
    """Python wrapper for SiluAndMul Triton kernel"""
    assert x.ndim == 2, "Input must be 2D"
    b, n = x.shape
    assert n % 2 == 0
    d = n // 2

    output = torch.empty((b, d), dtype=x.dtype, device=x.device)
    BLOCK_SIZE = 1024

    grid = (b, triton.cdiv(d, BLOCK_SIZE))
    silu_and_mul_kernel[grid](
        output, output.stride(0),
        x, x.stride(0),
        d,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return output


# 验证
x = torch.randn(128, 8192, device=device, dtype=torch.float16)  # 2 * 4096
result = triton_silu_and_mul(x)

# PyTorch 参考实现
d = x.shape[-1] // 2
expected = F.silu(x[:, :d]) * x[:, d:]

max_err = (result - expected).abs().max().item()
print(f"SiluAndMul - Max error: {max_err:.2e}")
assert max_err < 1e-2
print("SiluAndMul Triton kernel: PASSED!")
print(f"Input shape: {x.shape} -> Output shape: {result.shape}")

## 6. 将 SiluAndMul 注册为 CustomOp

In [ ]:
@SimpleCustomOp.register("my_silu_and_mul")
class MySiluAndMul(SimpleCustomOp):
    """
    完整的 SiluAndMul CustomOp 实现

    - forward_native: PyTorch 实现（CPU / torch.compile 兼容）
    - forward_cuda: Triton 实现（GPU 高性能）
    """

    def forward_native(self, x: torch.Tensor) -> torch.Tensor:
        """PyTorch-native implementation"""
        d = x.shape[-1] // 2
        return F.silu(x[..., :d]) * x[..., d:]

    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        """Triton implementation"""
        if x.ndim != 2:
            # vLLM 中的 token 总是 2D: (num_tokens, hidden_size)
            orig_shape = x.shape
            x = x.view(-1, x.shape[-1])
            result = triton_silu_and_mul(x)
            return result.view(orig_shape[:-1] + (result.shape[-1],))
        return triton_silu_and_mul(x)


# 测试注册和分发
op = MySiluAndMul()
print(f"Registered as: {op.name}")
print(f"Dispatch to: {op._forward_method.__name__}")

# 测试
x = torch.randn(128, 8192, device=device, dtype=torch.float16)
result = op(x)
d = x.shape[-1] // 2
expected = F.silu(x[:, :d]) * x[:, d:]
max_err = (result - expected).abs().max().item()
print(f"Max error: {max_err:.2e}")
assert max_err < 1e-2
print("\nMySiluAndMul CustomOp: PASSED!")
print(f"Registered ops: {list(op_registry.keys())}")

## 7. 编译配置控制

vLLM 提供了一个全局开关来控制哪些自定义算子启用/禁用：

```python
# vllm/model_executor/custom_op.py:271-304

@classmethod
def enabled(cls) -> bool:
    compilation_config = get_cached_compilation_config()
    custom_ops = compilation_config.custom_ops

    # 检查是否被显式启用/禁用
    enabled = f"+{cls.name}" in custom_ops    # "+silu_and_mul"
    disabled = f"-{cls.name}" in custom_ops   # "-silu_and_mul"

    return (CustomOp.default_on() or enabled) and not disabled
```

使用方式：
```python
# 全部启用（默认）
compilation_config = {"custom_ops": ["all"]}

# 全部禁用（使用 torch.compile 原生优化）
compilation_config = {"custom_ops": ["none"]}

# 禁用全部，但启用特定算子
compilation_config = {"custom_ops": ["none", "+silu_and_mul"]}

# 启用全部，但禁用特定算子
compilation_config = {"custom_ops": ["all", "-rms_norm"]}
```

这个设计让用户可以灵活地：
1. 在调试时禁用所有自定义算子，回退到 PyTorch 原生实现
2. 在使用 torch.compile 时只启用特定算子
3. 在特定硬件上禁用不兼容的算子

> **Source**: vllm/model_executor/custom_op.py:271-304

## 8. direct_register_custom_op: 另一种注册方式

除了继承 `CustomOp` 基类，vLLM 还提供了一个更轻量的注册函数，直接注册到 PyTorch 的算子库：

```python
# vllm/utils/torch_utils.py:758-797

def direct_register_custom_op(
    op_name: str,
    op_func: Callable,
    mutates_args: list[str] | None = None,
    fake_impl: Callable | None = None,
    target_lib: Library | None = None,
    dispatch_key: str | None = None,
):
    """
    绕过 torch.library.custom_op 的开销，直接注册算子。
    注册后可通过 torch.ops.vllm.op_name 调用。
    """
    schema_str = infer_schema(op_func, mutates_args=mutates_args)
    my_lib = target_lib or vllm_lib
    my_lib.define(op_name + schema_str)
    my_lib.impl(op_name, op_func, dispatch_key=dispatch_key)
```

两种注册方式的对比：

| 特性 | CustomOp 基类 | direct_register_custom_op |
|------|-------------|-------------------------|
| 多平台分发 | 自动 | 手动 |
| OOT 替换 | 支持 | 不支持 |
| 编译开关 | 支持 | 不支持 |
| 性能开销 | 略高 | 最小 |
| 适用场景 | 模型层算子 | 底层工具函数 |

> **Source**: vllm/utils/torch_utils.py:758-797

## 9. 端到端示例：在模型中使用 CustomOp

In [ ]:
class DemoLlamaMLP(nn.Module):
    """
    简化版 LlamaMLP，展示 CustomOp 在模型中的使用方式

    对应 vllm/model_executor/models/llama.py:81-121
    """
    def __init__(self, hidden_size=4096, intermediate_size=11008):
        super().__init__()
        # 在 Llama 中，gate_up_proj 是一个合并的线性层
        # 输出维度是 2 * intermediate_size（gate + up）
        self.gate_up_proj = nn.Linear(hidden_size, 2 * intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = MySiluAndMul()  # 使用我们注册的 CustomOp!

    def forward(self, x):
        # gate_up_proj: (batch, hidden) -> (batch, 2*intermediate)
        x = self.gate_up_proj(x)
        # act_fn: (batch, 2*intermediate) -> (batch, intermediate)
        x = self.act_fn(x)
        # down_proj: (batch, intermediate) -> (batch, hidden)
        x = self.down_proj(x)
        return x


# 创建一个小模型测试
torch.manual_seed(42)
model = DemoLlamaMLP(hidden_size=256, intermediate_size=512).half().to(device)
x = torch.randn(4, 256, device=device, dtype=torch.float16)

output = model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output dtype: {output.dtype}")
print(f"Activation function: {model.act_fn.name}")
print(f"\nUsing dispatch: {model.act_fn._forward_method.__name__}")

# 验证梯度可以正确回传（如果需要微调）
loss = output.sum()
loss.backward()
print(f"\nGradient check - gate_up_proj grad shape: {model.gate_up_proj.weight.grad.shape}")
print("Backward pass: PASSED!")

## 10. 本章小结

| 概念 | 说明 |
|------|------|
| `CustomOp` | 所有自定义算子的基类，提供多平台分发 |
| `@CustomOp.register("name")` | 注册内部算子到全局注册表 |
| `@CustomOp.register_oot(name="name")` | 注册外部替换算子 |
| `forward_native` | PyTorch 原生实现，CPU/编译器兼容 |
| `forward_cuda` | CUDA/Triton 高性能实现 |
| `dispatch_forward` | 构造时绑定方法，运行时零开销分发 |
| `enabled()` | 通过编译配置全局控制算子开关 |
| `direct_register_custom_op` | 轻量级注册到 torch.ops |

## 源码映射表

| 本教程实现 | vLLM 原始源码 | 章节 |
|-----------|-------------|------|
| `SimpleCustomOp` | `custom_op.py:103-354` | 2 |
| `MySiluAndMul` | `activation.py:117-148 (SiluAndMul)` | 6 |
| 注册装饰器 | `custom_op.py:307-320 (register)` | 2 |
| OOT 注册 | `custom_op.py:332-353 (register_oot)` | 2 |
| `DemoLlamaMLP` | `models/llama.py:81-121 (LlamaMLP)` | 9 |

## 下一步

在 [03-fused-operators.ipynb](03-fused-operators.ipynb) 中，我们将学习如何编写融合算子，将多个操作合并到一个 Triton kernel 中以提升性能。